In [1]:

import torch

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Select GPU runtime.")


PyTorch Version: 2.14.1+cu130
GPU Available: True
GPU Name: Tesla T4


In [3]:

!pip install -q fastapi uvicorn python-multipart pillow requests


In [4]:

import torch
import torchvision
from torchvision import models, transforms
from PIL import Image
import io
import time

# Select GPU or CPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Move model to GPU
model = model.to(device)
model.eval()

# Image preprocessing
preprocess = weights.transforms()

# Image classification labels
labels = weights.meta["categories"]

print("Model Loaded Successfully!")
print("Device:", device)
print("Model: ResNet18")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 74.6MB/s]


Model Loaded Successfully!
Device: cuda
Model: ResNet18


In [9]:


def predict_image(image):

    start_time = time.time()

    # Convert image to RGB
    image = image.convert("RGB")

    # Preprocess image
    input_tensor = preprocess(image).unsqueeze(0)

    # Move input to GPU
    input_tensor = input_tensor.to(device)

    # Run inference
    with torch.inference_mode():
        outputs = model(input_tensor)

    # Calculate probabilities
    probabilities = torch.nn.functional.softmax(
        outputs, dim=1
    )

    confidence, predicted = torch.max(
        probabilities, 1
    )

    prediction = labels[predicted.item()]

    execution_time = (time.time() - start_time) * 1000

    return {
        "prediction": prediction,
        "confidence": round(confidence.item() * 100, 2),
        "inference_time_ms": round(execution_time, 2),
        "device": str(device)
    }

print("Prediction Function Ready!")



Prediction Function Ready!


In [11]:

import threading
import uvicorn
import time
import requests

# Configure FastAPI server
config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="info"
)

api_server = uvicorn.Server(config)

# Start server in background
server_thread = threading.Thread(
    target=api_server.run,
    daemon=True
)

server_thread.start()

# Wait for server startup
time.sleep(5)

# Check server connection
try:
    response = requests.get(
        "http://127.0.0.1:8000/",
        timeout=10
    )

    print("Server Status:", response.status_code)
    print("Server Response:", response.json())

except requests.exceptions.RequestException as e:
    print("Server Error:", e)


INFO:     Started server process [791]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:51832 - "GET / HTTP/1.1" 200 OK
Server Status: 200
Server Response: {'message': 'Welcome to VisionPort', 'status': 'Online', 'model': 'ResNet18', 'device': 'cuda'}


In [12]:

from google.colab import files
import requests

uploaded = files.upload()

for filename in uploaded.keys():

    with open(filename, "rb") as image_file:

        response = requests.post(
            "http://127.0.0.1:8000/predict",
            files={
                "file": (
                    filename,
                    image_file,
                    "image/jpeg"
                )
            },
            timeout=60
        )

    print("\nPrediction Result:")
    print(response.status_code)
    print(response.json())


Saving traffic.jpg to traffic.jpg
INFO:     127.0.0.1:38124 - "POST /predict HTTP/1.1" 200 OK

Prediction Result:
200
{'filename': 'traffic.jpg', 'result': {'prediction': 'cab', 'confidence': 86.19, 'inference_time_ms': 1487.56, 'device': 'cuda'}}


In [14]:
print(requests.get(
    "http://127.0.0.1:8000/health"
).json())

INFO:     127.0.0.1:38880 - "GET /health HTTP/1.1" 200 OK
{'status': 'healthy', 'gpu_available': True, 'model_loaded': True}
